Since SmileyLlama is derived from a large language model, it inherits the ability to process natural language. Here we'll show a few examples of SmileyLlama responding to queries outside of natural language. More analysis of these can be found in the supplementary information in the SmileyLlama manuscript.

Here we'll begin by loading the model before running inference. As a side-note, even though we're running inference with greedy decoding, your results may differ on different hardware due to numerical imprecision, though the results generally hold. We've noticed some variation in results when e.g. asking about Christopher A Lipinski (on an 8xH100 node we tested, the model does not generate a SMILES string) and asking for a model with fewer than negative six H-bond donors (on a 1xA40 node, we noticed a different nonsensical molecule, but without the same repetitiveness)

In [1]:
from sltools.inference_tools import InferenceObject
temperature = 1.0
model_path = "THGLab/Llama-3.1-8B-SmileyLlama-1.1"
tokenizer_path = model_path
num_return_sequences = 1
max_new_tokens = 2048
io = InferenceObject(model_path, tokenizer_path, num_return_sequences, temperature, max_new_tokens)

PyTorch Using cuda device with 4 GPUs


2025-12-18 13:55:49.543307: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-18 13:56:01.735368: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-18 13:56:03.181568: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-18 13:56:03.877840: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-18 13:56:07.863082: I tensorflow/core/platform/cpu_feature_guar

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [2]:
system_text = "You are a helpful assistant"
user_text = "Write me some python code which calculates the n'th Fibonacci number"
prompts = [f"### Instruction:\n{system_text}\n\n### Input:\n{user_text}\n\n### Response:\n"]
raw_results = []
strings = io.generate_strings(prompts, generation_params={"do_sample":False, "max_new_tokens":128}, disable_tqdm=True)
print(strings[0][1][0])

/global/scratch/users/jmcavanagh/llchem/ana-env/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


def fibonacci(n):
    if n <= 0:
        return "Input should be a positive integer"
    elif n == 1:
        return 0
    elif n == 2:
        return 1
    else:
        a, b = 0, 1
        for _ in range(2, n):
            a, b = b, a + b
        return b


In [3]:
system_text = "You are a helpful assistant"
user_text = "Write me some python code which implements the mergesort of a list from scratch."
prompts = [f"### Instruction:\n{system_text}\n\n### Input:\n{user_text}\n\n### Response:\n"]
raw_results = []
strings = io.generate_strings(prompts, generation_params={"do_sample":False, "max_new_tokens":128}, disable_tqdm=True)
print(strings[0][1][0])

def merge_sort(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left, right):
    result = []
    while left and right:
        if left[0] <= right[0]:
            result.append(left.pop(0))
        else:
            result.append(right.pop(0))
    result.extend(left)
    result.extend(right)
    return result


In [4]:
system_text = "You are a helpful assistant"
user_text = "What is the molecular weight of hydrazine?"
prompts = [f"### Instruction:\n{system_text}\n\n### Input:\n{user_text}\n\n### Response:\n"]
raw_results = []
strings = io.generate_strings(prompts, generation_params={"do_sample":False, "max_new_tokens":128}, disable_tqdm=True)
print(strings[0][1][0])

N1N=C(C)C(C1)C


In [5]:
system_text = "You are a helpful assistant"
user_text = "Who was Christopher A. Lipinski in the field of chemistry? What was his \"Rule\"?"
prompts = [f"### Instruction:\n{system_text}\n\n### Input:\n{user_text}\n\n### Response:\n"]
raw_results = []
strings = io.generate_strings(prompts, generation_params={"do_sample":False, "max_new_tokens":128}, disable_tqdm=True)
print(strings[0][1][0])

Christopher A. Lipinski was a chemist who developed the "Lipinski's Rule of Five" which states that for a drug to be orally bioavailable, it should have the following properties: 
1) C(C)Cn1c2c(cccc2)c(c1)C(Nc1ccccc1)=O 
2) A molecular weight of 500 or less 
3) No more than 5 H-bond donors 
4) No more than 10 H-bond acceptors


In [6]:
system_text = "You are a helpful assistant"
user_text = "What is a Diels-Alder reaction in organic chemistry?"
prompts = [f"### Instruction:\n{system_text}\n\n### Input:\n{user_text}\n\n### Response:\n"]
raw_results = []
strings = io.generate_strings(prompts, generation_params={"do_sample":False, "max_new_tokens":128}, disable_tqdm=True)
print(strings[0][1][0])

A Diels-Alder reaction is a [4+2] cycloaddition between a diene and a dienophile.


Since SmileyLlama can take in prompts purely in natural language, it is possible to give SmileyLlama a nonsensical
instruction. Here, we ask SmileyLlama to generate molecules which are impossible, in principle, to generate.

Our first example will be a request with two properties which are impossible, in principle, for a molecule to satisfy. Our second will involve requesting a nonsensical property.

In [7]:
system_text = "You love and excel at generating SMILES strings of drug-like molecules"
user_text = "Output a SMILES string for a drug like molecule with the following properties: <= 3 H-bond acceptors, a substructure of c1c(OC)c(OC)c(OC)c(OC)c1:"
prompts = [f"### Instruction:\n{system_text}\n\n### Input:\n{user_text}\n\n### Response:\n"]
raw_results = []
strings = io.generate_strings(prompts, generation_params={"do_sample":False, "max_new_tokens":128}, disable_tqdm=True)
print(strings[0][1][0])

c1c(c(c(c(c1)OC)OC)OC)C1C2C(CCC=1)C1C(CCC=2)C1


In [8]:
system_text = "You love and excel at generating SMILES strings of drug-like molecules"
user_text = "Output a SMILES string for a drug like molecule with the following properties: <= -6 H-bond donors:"
prompts = [f"### Instruction:\n{system_text}\n\n### Input:\n{user_text}\n\n### Response:\n"]
raw_results = []
strings = io.generate_strings(prompts, generation_params={"do_sample":False, "max_new_tokens":128}, disable_tqdm=True)
print(strings[0][1][0])

c1c2c(ccc1)C(=O)N(C2=O)CC(Nc1ccc(cc1)C(=O)Nc1ccc(cc1)C(Nc1ccc(cc1)C(Nc1ccc(cc1)C(Nc1ccc(cc1)C(Nc1ccc(cc1)C(Nc1ccc(cc1)C(Nc1ccc(cc1)C(Nc1ccc(cc1)C(Nc1ccc(cc1)C(Nc1ccc(cc1)C(Nc1ccc(cc1)C(Nc1ccc(cc1)C(N
